# LangChain 核心用法（带看懂版）

> 目标：带你看懂 LangChain 的核心代码。全书用同一条真实场景贯穿：**电商客服订单问答**。
> 先看总览图，再用客服问答链串起核心概念，看懂两种写法，最后逐个看懂核心能力。
> 基于 langchain 1.3.15 / langchain-core 1.5.6（2026-08-18）验证；无 Key 示例用 fake 模型模拟。

## 框架总览：一张图看懂 LangChain

LangChain 是一套**组件库 + 管道语言**：提示词、模型、解析器都是 Runnable 组件，用 LCEL 管道符 `|` 串成一条链。

```mermaid
flowchart LR
    A[用户输入] --> B[ChatPromptTemplate 提示词模板]
    B -->|LCEL 管道 | C[ChatModel 模型]
    C --> D[OutputParser 输出解析]
    D --> E[最终回答]
    C -.bind_tools 工具调用.-> F[Tool 工具]
    B -.RunnableWithMessageHistory 会话记忆.-> G[会话历史]
    C -.stream 流式.-> H[逐块输出]
```

先记住四个词：**Runnable（统一接口）、LCEL（管道）、ChatPromptTemplate（模板）、ChatModel（模型）**。

## 核心概念（先读一遍，遇到再回来看）

| 概念 | 作用 | 在哪看 |
| --- | --- | --- |
| Runnable | 所有组件的统一接口：`invoke` / `stream` / `batch` | 概念关系、能力 3 |
| LCEL | 管道符 `|`，把 Runnable 串成链 | 概念关系、能力 1 |
| ChatPromptTemplate | 提示词模板，拼装系统消息与用户输入 | 概念关系、能力 2 |
| ChatModel | 模型统一接口，可切换厂商 | 概念关系、能力 3 |
| OutputParser | 输出解析，把模型回复变成字符串或对象 | 概念关系、能力 2 |
| Memory | 会话记忆，把历史塞回提示词 | 能力 4 |
| stream | 流式输出，逐块返回 | 能力 5 |
| bind_tools / create_agent | 工具调用与高层 Agent 构建 | 能力 6 |

## 概念之间的关系：用客服问答链全部串起来

把上一张表里的全部概念放进同一条**客服订单问答链**：

```mermaid
flowchart LR
    A[用户问题] --> B[ChatPromptTemplate 客服话术模板]
    B -->|管道符| C[ChatModel 客服大脑]
    C --> D[OutputParser 整理回答]
    D --> E[回答用户]
    C -.bind_tools 查询订单.-> F[订单查询工具]
    B -.Memory 记住历史.-> G[会话历史]
    C -.stream 逐字输出.-> H[前端打字机效果]
```

| 概念 | 在客服链里的位置 |
| --- | --- |
| Runnable | prompt、模型、parser 都是 Runnable，统一 `invoke` 调用 |
| LCEL | `prompt | model | parser` 管道符把三者串成一条链 |
| ChatPromptTemplate | 客服话术模板：系统人设 + 用户问题 |
| ChatModel | 客服大脑，这里用 fake 模型模拟真实模型 |
| OutputParser | 把模型的回复消息变成纯字符串 |
| Memory | 记住用户上一句，多轮对话不断上下文（能力 4） |
| stream | 回答逐字流到前端（能力 5） |
| bind_tools / create_agent | 让模型能查订单、组成完整 Agent（能力 6） |

先运行下面的**骨架链**（不需要 API Key，用 fake 模型模拟），建立最小直觉。

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


# fake 模型：不调 API，固定返回这条消息，用于本地看懂管道机制
fake = FakeMessagesListChatModel(responses=[AIMessage(content="订单已发货，预计明天送达")])

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电商客服，用一句话回答用户问题。"),
    ("human", "{question}"),
])

# LCEL：管道符把模板、模型、解析器串成一条链
chain = prompt | fake | StrOutputParser()

result = chain.invoke({"question": "我的订单 A001 发货了吗？"})
print(result)


### 骨架链：按执行顺序走一遍

| 顺序 | 执行的代码 | 发生了什么 |
| --- | --- | --- |
| 1 | `chain.invoke({"question": "..."})` | 输入进入管道 |
| 2 | `prompt` 把 system + 用户问题组装成消息 | 传给模型 |
| 3 | fake 模型返回固定 `AIMessage` | 模拟真实模型的回复 |
| 4 | `StrOutputParser` 提取消息的文本 | 得到最终字符串 |

这条链只是**一条直线**。LangChain 的真正能力在后面的「核心能力逐个看」：批处理、结构化解析、记忆、流式、工具调用。

## 先看懂两种写法

同一个客服问答任务，LangChain 有两种主流写法：
用法一用 LCEL 管道手搓链，用法二用官方高层 `create_agent`。

### 用法一：LCEL 手搓链（最透明）

先看这张管道图，再看代码：

```mermaid
flowchart LR
    A[invoke 用户问题] --> B[prompt 组装消息]
    B --> C[fake 模型返回回复]
    C --> D[StrOutputParser 提取文本]
    D --> E[print 输出]
```

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


fake = FakeMessagesListChatModel(responses=[AIMessage(content="A001 已发货")])

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电商客服。"),
    ("human", "{question}"),
])

chain = prompt | fake | StrOutputParser()

result = chain.invoke({"question": "订单 A001 发货了吗？"})
print(result)


#### 用法一：按执行顺序拆解

| 顺序 | 执行的代码 | 结果 |
| --- | --- | --- |
| 1 | `chain.invoke({"question": "..."})` | 输入进入管道 |
| 2 | `prompt` 填充模板，生成消息列表 | `[system 客服, human 问题]` |
| 3 | fake 模型返回 `AIMessage(content="A001 已发货")` | 模拟模型回复 |
| 4 | `StrOutputParser` 提取 `content` | 输出 `A001 已发货` |

管道里每个环节都是 Runnable，`|` 只是把它们按顺序接起来。

### 用法二：create_agent（官方高层构建）

需要 `OPENAI_API_KEY`，没 Key 先看结构。官方定位：**Agent = Model + Harness**，harness 是模型外围的一切（提示词、工具、中间件）：

```mermaid
flowchart LR
    A[Agent] --> B[Model 模型]
    A --> C[Harness 外壳]
    C --> D[system_prompt 人设]
    C --> E[tools 工具]
    C --> F[中间件 middleware]
```

In [ ]:
from langchain.agents import create_agent


def check_stock(item: str) -> str:
    """查询商品是否有货。"""
    return f"{item} 有货"


agent = create_agent(
    model="openai:gpt-5.5",
    tools=[check_stock],
    system_prompt="你是电商客服助手",
)

result = agent.invoke({"messages": [{"role": "user", "content": "键盘有货吗？"}]})
print(result["messages"][-1].content)


#### 用法二：按执行顺序拆解（ReAct 循环）

| 顺序 | 执行内容 | 说明 |
| --- | --- | --- |
| 1 | `agent.invoke(用户消息)` | 把问题发给模型 |
| 2 | 模型返回 tool_call：调用 `check_stock("键盘")` | 模型决定需要工具 |
| 3 | 框架执行工具，返回「键盘 有货」 | 结果喂回模型 |
| 4 | 模型返回最终回答 | 循环结束 |

LangChain 的 agent 构建在 LangGraph 之上，所以它天然具备持久化、人工介入等能力。

## 两种写法怎么选

| 写法 | 代码量 | 控制力 | 适用场景 |
| --- | --- | --- | --- |
| LCEL 手搓链 | 少 | 中 | 固定流水线、RAG、结构化输出 |
| create_agent | 最少 | 低 | 标准工具调用 Agent，快速交付 |
| LangGraph | 多 | 最强 | 复杂图、循环、人工介入、多 Agent |

AI 会帮你写代码，但选哪种写法、什么时候升级到 LangGraph，需要你能看懂管道和 Agent 的区别。

## 核心能力逐个看

骨架链是「一条直线」。下面六个能力才是 LangChain 常用但不一定一眼看懂的机制，全部沿用客服问答场景。

### 能力 1：LCEL 组合与批处理

`|` 可以任意组合 Runnable，`batch` 一次处理多条输入。真实场景：客服系统同时收到很多用户提问，批量交给链处理。

```mermaid
flowchart LR
    A[用户问题 1] --> B[prompt]
    A2[用户问题 2] --> B
    B --> C[fake 模型]
    C --> D[StrOutputParser]
    D --> E[回答 1]
    D --> F[回答 2]
```

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


fake = FakeMessagesListChatModel(responses=[
    AIMessage(content="A001 已发货"),
    AIMessage(content="退货需在 7 天内申请"),
])

prompt = ChatPromptTemplate.from_messages([("human", "{question}")])
chain = prompt | fake | StrOutputParser()

results = chain.batch([
    {"question": "订单 A001 发货了吗？"},
    {"question": "怎么退货？"},
])
for q, r in zip(["A001 发货了吗？", "怎么退货？"], results):
    print(f"{q} -> {r}")


批处理的拆解：

- `chain.batch([...])` 一次喂两条输入，`|` 管道对每条输入分别走一遍
- fake 模型按顺序消费 `responses` 列表，两条输入各拿到一条回复
- 真实模型会并行请求，批量比循环调用更快

### 能力 2：提示词模板 + 结构化解析

输出解析器能把模型回复解析成带类型的对象，而不是裸字符串。真实场景：客服查询订单后，把结果解析成结构化的订单信息。

```mermaid
flowchart LR
    A[查询订单] --> B[prompt]
    B --> C[模型返回 JSON 文本]
    C --> D[PydanticOutputParser 校验解析]
    D --> E[OrderInfo 对象]
```

In [ ]:
from pydantic import BaseModel, Field

from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate


class OrderInfo(BaseModel):
    """订单信息结构。"""
    order_id: str = Field(description="订单号")
    status: str = Field(description="订单状态")


# fake 模型直接返回一段 JSON 文本
fake = FakeMessagesListChatModel(
    responses=[AIMessage(content='{"order_id": "A001", "status": "已发货"}')]
)

prompt = ChatPromptTemplate.from_messages([("human", "{question}")])
parser = PydanticOutputParser(pydantic_object=OrderInfo)
chain = prompt | fake | parser

obj = chain.invoke({"question": "查询订单 A001"})
print(f"order_id={obj.order_id}, status={obj.status}")
print(f"类型：{type(obj).__name__}")


结构化解析的拆解：

- `OrderInfo` 定义了输出结构：order_id + status
- fake 模型返回 JSON 字符串，`PydanticOutputParser` 校验并解析
- 结果不是字典而是 `OrderInfo` 对象，字段带类型，后续代码可以直接用 `obj.order_id`

### 能力 3：模型统一接口

所有 ChatModel 都提供同一个接口：`invoke` / `stream` / `batch`。真实场景：切换模型厂商（OpenAI → Anthropic → Gemini）时业务代码不用改。

```mermaid
flowchart TB
    A[ChatModel 统一接口] --> B[invoke 单次调用]
    A --> C[stream 流式输出]
    A --> D[batch 批量调用]
```

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage, HumanMessage


model = FakeMessagesListChatModel(responses=[AIMessage(content="接口一致")])

print("invoke:", model.invoke([HumanMessage(content="你好")]).content)
print("stream:", [c.content for c in model.stream([HumanMessage(content="你好")])])
print("batch:", [m.content for m in model.batch([[HumanMessage(content="你好")]])])

# 真实切换模型：init_chat_model("anthropic:claude-sonnet-4-6")，需 API Key


统一接口的拆解：

- 同一个模型对象支持 `invoke`、`stream`、`batch` 三种调用，签名一致
- 真实项目用 `init_chat_model("openai:gpt-5.5")` 或 `"anthropic:..."` 切换厂商，下游代码不需要改
- fake 模型的 stream 一次返回完整内容；真实模型会逐 token 返回

### 能力 4：会话记忆

默认链是无状态的，`RunnableWithMessageHistory` 把历史消息塞回提示词，实现多轮对话。真实场景：用户问完订单又追问物流，客服要记得上下文。

```mermaid
flowchart LR
    A[用户输入] --> B[prompt]
    G[会话历史] --> B
    B --> C[模型]
    C --> D[回答]
    D --> G
```

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory


fake = FakeMessagesListChatModel(responses=[AIMessage(content="好的，我记住了")])

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是客服。"),
    ("placeholder", "{history}"),
    ("human", "{input}"),
])

chain = prompt | fake | StrOutputParser()
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=lambda session_id: InMemoryChatMessageHistory(),
    input_messages_key="input",
    history_messages_key="history",
)

result = chain_with_history.invoke(
    {"input": "你好"}, {"configurable": {"session_id": "customer-1"}}
)
print(result)


会话记忆的拆解：

- `get_session_history` 按 `session_id` 返回历史存储对象
- 每次调用前，历史会注入 `{history}` 占位符，回答后新消息也会写回历史
- **注意**：`RunnableWithMessageHistory` 已被官方标记 deprecated，生产环境推荐用 LangGraph 的持久化（thread + checkpointer）

### 能力 5：流式输出

`stream` 让结果逐块到达，而不是等全部算完。真实场景：前端打字机效果，用户不用干等。

```mermaid
flowchart LR
    A[chain.stream] --> B[块1 订]
    B --> C[块2 单]
    C --> D[块3 已]
    D --> E[块4 发]
    E --> F[块5 货]
```

In [ ]:
from langchain_core.runnables import RunnableGenerator


def answer_stream(input):
    """模拟模型逐字输出回答。"""
    for ch in ["订", "单", "已", "发", "货"]:
        yield ch


chain = RunnableGenerator(answer_stream)

for chunk in chain.stream({}):
    print(chunk, end=" ")
print()


流式的拆解：

- `RunnableGenerator` 把生成器函数变成 Runnable，`stream` 逐块拿到 yield 的值
- 真实场景是 `model.stream(...)`：模型逐 token 输出，前端边收边显示
- 流式和 invoke 是同一个接口的两种模式，链的写法不用变

### 能力 6：工具绑定与 create_agent

让模型调用工具需要 `bind_tools`；`create_agent` 把模型 + 工具 + 人设打包成完整 Agent。真实场景：客服 Agent 自己决定要不要查订单。

```mermaid
flowchart LR
    A[用户问题] --> B[模型]
    B -->|要查订单| C[订单工具]
    C --> B
    B -->|直接回答| D[最终回答]
```

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool


@tool
def query_order(order_id: str) -> str:
    """查询订单状态。"""
    return f"订单 {order_id} 已发货"


model = ChatOpenAI(model="gpt-4o-mini").bind_tools([query_order])

# 也可以直接用 create_agent 一步打包：模型 + 工具 + 人设
agent = create_agent(
    model=model,
    tools=[query_order],
    system_prompt="你是电商客服，需要时查询订单。",
)

response = agent.invoke({"messages": [{"role": "user", "content": "A001 到哪了？"}]})
print(response["messages"][-1].content)


工具调用的拆解：

- `@tool` 把函数声明成工具；`bind_tools` 把工具绑定到模型
- 模型返回 tool_call 时，框架执行工具并把结果喂回模型，形成 ReAct 循环
- `create_agent` 是更高层封装：模型、工具、人设一次打包
- 这个能力需要真实模型（API Key），代码结构与 LangGraph 的 create_react_agent 对应

## 核心能力总览

| 能力 | 客服场景 | 关键 API |
| --- | --- | --- |
| 组合与批处理 | 多条提问一次处理 | `|` / `batch` |
| 结构化解析 | 订单信息解析成对象 | `PydanticOutputParser` |
| 统一接口 | 换模型不改代码 | `invoke` / `stream` / `batch` |
| 会话记忆 | 多轮对话记住上下文 | `RunnableWithMessageHistory`（已弃用） |
| 流式输出 | 前端打字机效果 | `stream` / `RunnableGenerator` |
| 工具与 Agent | 客服自主查订单 | `bind_tools` / `create_agent` |